In [1]:
import os
import numpy as np

In [2]:
PROC_DIR = '../data/processed'
VOCAB_SIZE = 1001
MAX_SEQ_LEN = 10
BATCH_SIZE = 256
SEED = 42

In [3]:
X_padded = np.load(f'{PROC_DIR}/X_padded.npy').astype(np.float32)
Y = np.load(f'{PROC_DIR}/y_target.npy').astype(np.float32)

assert X_padded.ndim == 3 and X_padded.shape[1:] == (MAX_SEQ_LEN, VOCAB_SIZE)
assert Y.shape == (X_padded.shape[0], VOCAB_SIZE)

print(f"X_padded shape: {X_padded.shape}")
print(f"Y shape: {Y.shape}")

X_padded shape: (14074, 10, 1001)
Y shape: (14074, 1001)


In [4]:
S = np.concatenate([X_padded, Y[:, None, :]], axis=1)
print(f"Full sequences shape (history + test target): {S.shape}")

Full sequences shape (history + test target): (14074, 11, 1001)


In [5]:
def build_examples(S, p):
    target = S[:, p, :]
    has_target = target.sum(axis=1) > 0
    has_input = S[:, :p, :].sum(axis=(1, 2)) > 0
    users = np.flatnonzero(has_target & has_input)
    return np.stack([users, np.full(users.size, p)], axis=1)

In [6]:
T = MAX_SEQ_LEN
train_examples = np.concatenate([build_examples(S, p) for p in range(1, T - 1)], axis=0)
val_examples = build_examples(S, T - 1)
test_examples = build_examples(S, T)

print(f"Train examples: {len(train_examples)}")
print(f"Validation examples: {len(val_examples)}")
print(f"Test examples: {len(test_examples)}")

Train examples: 38981
Validation examples: 14074
Test examples: 14074


In [7]:
assert np.all(test_examples[:, 1] == T)
assert np.all(val_examples[:, 1] == T - 1)
assert np.all(train_examples[:, 1] <= T - 2)
print("Leakage check passed: test and validation targets are not used as training targets")

Leakage check passed: test and validation targets are not used as training targets


In [8]:
def window_for(S, u, p, max_seq_len=MAX_SEQ_LEN):
    L = min(p, max_seq_len)
    x = np.zeros((max_seq_len, S.shape[2]), dtype=np.float32)
    x[max_seq_len - L:] = S[u, p - L:p]
    return x

In [9]:
class SequenceBatcher:
    def __init__(self, S, examples, batch_size=BATCH_SIZE, shuffle=True, seed=SEED, max_seq_len=MAX_SEQ_LEN):
        if len(examples) == 0:
            raise ValueError('no examples for this split')
        self.S = S
        self.examples = examples
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.max_seq_len = max_seq_len
        self.rng = np.random.default_rng(seed)

    def __len__(self):
        return int(np.ceil(len(self.examples) / self.batch_size))

    def __iter__(self):
        order = np.arange(len(self.examples))
        if self.shuffle:
            self.rng.shuffle(order)
        for start in range(0, len(order), self.batch_size):
            idx = order[start:start + self.batch_size]
            pairs = self.examples[idx]
            X = np.stack([window_for(self.S, u, p, self.max_seq_len) for u, p in pairs])
            y = self.S[pairs[:, 0], pairs[:, 1]].astype(np.float32)
            yield X, y

In [10]:
train_batcher = SequenceBatcher(S, train_examples, shuffle=True)
val_batcher = SequenceBatcher(S, val_examples, shuffle=False)
test_batcher = SequenceBatcher(S, test_examples, shuffle=False)

X_b, y_b = next(iter(train_batcher))
assert X_b.ndim == 3
assert X_b.shape[1:] == (MAX_SEQ_LEN, VOCAB_SIZE)
assert y_b.shape[1:] == (VOCAB_SIZE,)
print(f"Batch X shape: {X_b.shape}, y shape: {y_b.shape}")
print(f"Batches per epoch - train: {len(train_batcher)}, validation: {len(val_batcher)}, test: {len(test_batcher)}")

Batch X shape: (256, 10, 1001), y shape: (256, 1001)
Batches per epoch - train: 153, validation: 55, test: 55


In [11]:
import tensorflow as tf

signature = (
    tf.TensorSpec(shape=(None, MAX_SEQ_LEN, VOCAB_SIZE), dtype=tf.float32),
    tf.TensorSpec(shape=(None, VOCAB_SIZE), dtype=tf.float32),
)

train_ds = tf.data.Dataset.from_generator(
    lambda: iter(SequenceBatcher(S, train_examples, shuffle=True)),
    output_signature=signature
).prefetch(tf.data.AUTOTUNE)

val_ds = tf.data.Dataset.from_generator(
    lambda: iter(SequenceBatcher(S, val_examples, shuffle=False)),
    output_signature=signature
).prefetch(tf.data.AUTOTUNE)

test_ds = tf.data.Dataset.from_generator(
    lambda: iter(SequenceBatcher(S, test_examples, shuffle=False)),
    output_signature=signature
).prefetch(tf.data.AUTOTUNE)

for xb, yb in train_ds.take(1):
    print(f"tf.data batch X: {xb.shape}, y: {yb.shape}")

tf.data batch X: (256, 10, 1001), y: (256, 1001)


In [12]:
np.savez(
    f'{PROC_DIR}/splits.npz',
    train=train_examples,
    validation=val_examples,
    test=test_examples
)
print(f"Saved splits to {PROC_DIR}/splits.npz")

Saved splits to ../data/processed/splits.npz
